<h2>Identify, validate, and correct data entry errors using advanced regular expressions (regex) patterns and automated validation rules, focusing on fields prone to inconsistencies such as dates, locations, and numeric ranges.</h2>

In [2]:
import pandas as pd
import re

In [18]:
sales_data = pd.DataFrame({
    'created_at': ['2026-02-28', '2026/08/24', '2026-02-31', 'invalid_date'],
    'location_code': ['NY_10001', 'ca-90210', 'FL12345', 'INVALID_LOC'],
    'selling_price': ['25.00', '-10.50', 'FREE', '150.00'],
    'mrp': [30.00, 20.00, 50.00, 100.00]
})

In [19]:
date_pattern = r'^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$'
loc_pattern = r'^[A-Z]{2}[_-]?\d{5}$'
num_pattern = r'^\d+(\.\d{1,2})?$'

In [20]:
cleaned_rows = []

In [25]:
for idx, row in sales_data.iterrows():
   
    raw_date = str(row['created_at']).strip().replace('/', '-')
    valid_date = bool(re.match(date_pattern, raw_date))
    if valid_date:
        try:
            pd.to_datetime(raw_date, format='%Y-%m-%d', errors='raise')
        except ValueError:
            valid_date = False

    raw_loc = str(row['location_code']).strip().upper()
    valid_loc = bool(re.match(loc_pattern, raw_loc))

    raw_sp = str(row['selling_price']).strip()
    valid_sp_format = bool(re.match(num_pattern, raw_sp))
    
    valid_range = False
    if valid_sp_format:
        sp = float(raw_sp)
        mrp = float(row['mrp'])
        valid_range = (sp > 0) and (sp <= mrp)

    is_fully_valid = valid_date and valid_loc and valid_sp_format and valid_range

    cleaned_rows.append({
        'original_date': row['created_at'],
        'clean_date': raw_date if valid_date else 'ERROR',
        'clean_location': raw_loc if valid_loc else 'ERROR',
        'clean_price': float(raw_sp) if (valid_sp_format and valid_range) else 'ERROR',
        'is_valid_entry': is_fully_valid
    })

results_df = pd.DataFrame(cleaned_rows)
print(results_df)

  original_date  clean_date clean_location clean_price  is_valid_entry
0    2026-02-28  2026-02-28       NY_10001        25.0            True
1    2026/08/24  2026-08-24       CA-90210       ERROR           False
2    2026-02-31       ERROR        FL12345       ERROR           False
3  invalid_date       ERROR          ERROR       ERROR           False
4    2026-02-28  2026-02-28       NY_10001        25.0            True
5    2026/08/24  2026-08-24       CA-90210       ERROR           False
6    2026-02-31       ERROR        FL12345       ERROR           False
7  invalid_date       ERROR          ERROR       ERROR           False
